# WorldLoop · 5-Minute Quickstart

**模型只提议，世界裁决。** WorldLoop is an environment-authoritative multi-agent simulation and trajectory-data system: policies (or LLMs) only submit candidate actions — the world validates, settles, and records every transition as a verifiable state diff + hash.

This notebook walks through the core loop in five minutes:

1. Compile a world from a YAML scenario
2. Observe the world state
3. Propose an action → the world adjudicates
4. The world *rejects* unknown proposals
5. Multi-step run + hash chain + counterfactual branching

```python
# if WorldLoop is not installed yet:
# pip install worldloop-kernel worldloop-scenarios
```

In [ ]:
from worldloop_kernel import ActionProposal, hash_state
from worldloop_scenarios import compile_file

# Resolve the scenario file (works from the repo root or the examples/ dir).
import os
from pathlib import Path

candidates = [
    Path("examples/discrete_grid.yaml"),          # repo root (published)
    Path("current/worldloop-scenarios/examples/discrete_grid.yaml"),
    Path("discrete_grid.yaml"),                    # cwd == examples/
]
scenario_path = next((p for p in candidates if p.exists()), None)
assert scenario_path is not None, "discrete_grid.yaml not found"

package = compile_file(scenario_path)
world = package.world_factory(seed=42)
print("compiled", package.spec.scenario.scenario_id, "✓")

## 1 · Observe the world

`world.observe()` returns a **formal state view** — entities (position, energy, alive) and meta (tick, scenario, hash). This is `S_t`, the state the world actually settles against, not a model's self-description.

In [ ]:
before = world.observe()
print("tick:", before.meta.tick)
print("agents:", before.entities.ids)
print("x/y  :", list(before.entities.columns["x"]), list(before.entities.columns["y"]))
print("energy:", [round(e, 2) for e in before.entities.columns["energy"]])

## 2 · Propose → the world adjudicates

A policy (or an LLM) submits an `ActionProposal`. The world validates it, settles its effects, and returns a transition record with a **hash chain**: `state_after_hash[t] == state_before_hash[t+1]`.

In [ ]:
proposal = ActionProposal(
    agent_id=before.entities.ids[0],
    action_type="forage",
    params={},
    proposed_at_tick=before.meta.tick,
    proposer="quickstart",
)

executed, receipt = world.validate_action(proposal)
record = world.step(executed)
after = world.observe()

print("outcome :", receipt.outcome_code, "| success:", receipt.success)
print("tick    :", before.meta.tick, "->", after.meta.tick)
print("hash ok :", record.state_after_hash == hash_state(after))
print("hash    :", record.state_after_hash[:20], "...")
print("agent 0 energy:", round(before.entities.columns["energy"][0], 2),
      "->", round(after.entities.columns["energy"][0], 2))

## 3 · The world rejects what it doesn't know

An LLM is free to *propose anything* — the world is the one that says no. Propose an action type that isn't in the scenario and watch it fail closed with zero state change.

In [ ]:
state_before = world.observe()
bad = ActionProposal(
    agent_id=state_before.entities.ids[0],
    action_type="fly",
    params={"to": "moon"},
    proposed_at_tick=state_before.meta.tick,
    proposer="rogue-llm",
)
executed, receipt = world.validate_action(bad)
record = world.step(executed)
state_after = world.observe()

print("outcome_code:", receipt.outcome_code)
print("success     :", receipt.success)
print("energy     :",
      state_before.entities.columns["energy"] == state_after.entities.columns["energy"],
      "(unchanged)")
print("position   :",
      state_before.entities.columns["x"] == state_after.entities.columns["x"]
      and state_before.entities.columns["y"] == state_after.entities.columns["y"],
      "(unchanged)")
print("tick       :", state_after.meta.tick, "(meta only)")

## 4 · Multi-step run & deterministic hash chain

Run a few steps and collect the hash chain. Because the world is deterministic given a seed, **any two independent runs produce byte-identical hashes** — that's the reproducibility guarantee behind audit, debugging, and trustworthy training data.

In [ ]:
def run_episode(seed=42, steps=10):
    w = package.world_factory(seed=seed)
    hashes = []
    for _ in range(steps):
        st = w.observe()
        proposal = ActionProposal(
            agent_id=st.entities.ids[0], action_type="forage", params={},
            proposed_at_tick=st.meta.tick, proposer="quickstart",
        )
        executed, _ = w.validate_action(proposal)
        rec = w.step(executed)
        hashes.append(rec.state_after_hash)
    return hashes

run_a = run_episode(seed=42, steps=10)
run_b = run_episode(seed=42, steps=10)

print("identical across runs:", run_a == run_b)
print("chain links:")
for i, h in enumerate(run_a):
    print(f"  t={i+1:>2}  {h[:24]}...")

## 5 · Counterfactual branching

From the **same checkpoint**, fork the world into two branches and settle them independently. Same history, different choices, two comparable worlds — the infrastructure for counterfactual research and data augmentation.

In [ ]:
from worldloop_kernel import ActionProposal

def settle(w, action_type, steps):
    for _ in range(steps):
        st = w.observe()
        proposal = ActionProposal(
            agent_id=st.entities.ids[0], action_type=action_type, params={},
            proposed_at_tick=st.meta.tick, proposer="branch",
        )
        executed, _ = w.validate_action(proposal)
        w.step(executed)
    return w.observe().entities.columns["energy"][0]

# Common history: 5 forage steps, then checkpoint.
common = package.world_factory(seed=42)
settle(common, "forage", 5)
ckpt = common.checkpoint()

# Fork into two worlds from the exact same state.
branch_a = package.world_factory(seed=42); branch_a.restore(ckpt)
branch_b = package.world_factory(seed=42); branch_b.restore(ckpt)

ea = settle(branch_a, "forage", 5)   # keep foraging
eb = settle(branch_b, "rest", 5)     # rest instead
print(f"branch A (forage x5): energy = {ea:.2f}")
print(f"branch B (rest   x5): energy = {eb:.2f}")
print(f"treatment effect    : {ea - eb:+.2f}")

## 6 · Next: the emergency scheduling demo

The `examples/emergency_demo.yaml` scenario drives a four-role team (leader / gatherer / comms / patrol) against a rising hazard gauge, with an ASCII-animated CLI:

```bash
python examples/demo/emergency_demo.py
```

and the animated GIF you saw in the README is generated by:

```bash
python examples/make_emergency_gif.py
```

---
**Want more?**

- More scenarios: `examples/*.yaml` (grid, continuous field, graph, market, emergency)
- External envs: `worldloop-adapters` (PettingZoo / Gymnasium / OpenEnv)
- Data pipeline: `worldloop-data` (rollouts, counterfactuals, leakage checks, export)
- Evidence & boundaries: see the repository `README.md` — we report negative results too.